# Lithium Server — Integration Tests

Uses the LangGraph SDK against the server running locally in Docker.

```bash
docker build -t lithium .
docker run --rm -p 8000:8000 \
  -e ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY \
  -e API_KEY=test-key \
  lithium
```

In [29]:
from pprint import pprint

In [30]:
from langgraph_sdk import get_client

client = get_client(
    url="http://localhost:8000",
    api_key="test-key",  # must match API_KEY env var passed to the container
)

## Health check

In [31]:
import httpx

response = httpx.get("http://localhost:8000/ok")
response.raise_for_status()
print(response.json())

{'status': 'ok'}


## List assistants

In [32]:
assistants = await client.assistants.search(graph_id="orient")
print(assistants)

assistant_id = assistants[0]["assistant_id"]

[{'assistant_id': 'orient', 'graph_id': 'orient', 'config': {}, 'context': {}, 'created_at': '2026-04-06T22:51:14.050049+00:00', 'updated_at': '2026-04-06T22:51:14.050052+00:00', 'metadata': {}, 'version': 1, 'name': 'orient', 'description': None}]


## Auth check — rejected request

In [33]:
bad_client = get_client(url="http://localhost:8000", api_key="wrong-key")
try:
    await bad_client.assistants.search()
    print("ERROR: expected rejection")
except Exception as e:
    print(f"Correctly rejected: {e}")

ERROR: expected rejection


## Stateless run (wait)

In [34]:
result = await client.runs.wait(
    thread_id=None,
    assistant_id=assistant_id,
    input={"messages": [{"role": "human", "content": "What is a problem statement?"}]},
)

for msg in result["messages"]:
    role = msg.get("type", msg.get("role", "?"))
    content = msg.get("data", {}).get("content") or msg.get("content", "")
    print(f"[{role}] {content[:300]}\n")

HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/runs/wait'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

## Stateless run (streaming)

In [ ]:
async for chunk in client.runs.stream(
    thread_id=None,
    assistant_id=assistant_id,
    input={"messages": [{"type": "human", "content": "Briefly describe what you help with."}]},
    stream_mode="values",
):
    pprint(chunk)

StreamPart(event='metadata', data={'run_id': '87bf3552-2edc-46e6-9933-22a614e3ee17'}, id=None)
StreamPart(event='values', data={'messages': [{'type': 'human', 'data': {'content': 'Briefly describe what you help with.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'be6c4046-ecdf-4284-97fa-4a6e4f6e3c1c'}}]}, id=None)
StreamPart(event='error', data={'error': 'AttributeError', 'message': "'NoneType' object has no attribute 'model'"}, id=None)
StreamPart(event='end', data=None, id=None)


## Stateful multi-turn conversation

Create a thread and send multiple turns — the server accumulates message history.

In [ ]:
thread = await client.threads.create()
thread_id = thread["thread_id"]
print(f"Created thread: {thread_id}")

Created thread: 0cb912de-69f1-4e72-aa48-d520665bedcd


In [ ]:
# Turn 1
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={"messages": [{"type": "human", "content": "I'm building a tool to help engineers write better incident post-mortems."}]},
)

last = result["messages"][-1]
print(f"[turn 1] {last.get('data', {}).get('content') or last.get('content', '')}")

HTTPStatusError: Server error '500 Internal Server Error' for url 'http://localhost:8000/threads/0cb912de-69f1-4e72-aa48-d520665bedcd/runs/wait'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

In [ ]:
# Turn 2 — server remembers the conversation
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={"messages": [{"type": "human", "content": "The main users are on-call engineers at mid-sized SaaS companies."}]},
)

last = result["messages"][-1]
print(f"[turn 2] {last.get('data', {}).get('content') or last.get('content', '')}")

In [ ]:
# Inspect full thread state
state = await client.threads.get_state(thread_id)
print(f"Message count in thread: {len(state['values'].get('messages', []))}")

## Cleanup

In [ ]:
await client.threads.delete(thread_id)
print(f"Deleted thread {thread_id}")